# Toy 4-knopen walkthrough — assignment-stappen met handmatige getallen

Dit notebook is een **minimaal voorbeeld** (4 knopen) dat dezelfde pipeline doorloopt als `insider_threat_bn_prototype.ipynb` (8 knopen). Alle kerngetallen uit de uitleg zijn hier reproduceerbaar in pyAgrum.

### Koppeling met de assignment

| Component | Assignment | Dit notebook |
|-----------|------------|--------------|
| **Taak 1** | Handmatig BN, inferentie, validatie | Secties 2–4 |
| **Taak 2.2a** | Structuur leren (HC + MIIC) | Sectie 5 |
| **RQ4** | Noisy-OR op uitkomstknoop | Sectie 3 + 6 |
| **Taak 2.2b** | Classificatie original / learned / naive Bayes | Sectie 7 |

### Speelgoed-DAG

```
Dissatisfaction → BadBehaviour → Incident
WeakTechControls  ─────────────→ Incident
```

| Knoop | Betekenis (vereenvoudigd) |
|-------|---------------------------|
| `Dissatisfaction` | Medewerker ontevreden |
| `BadBehaviour` | Zorgwekkend gedrag gezien |
| `WeakTechControls` | Technische controles zwak |
| `Incident` | Insider-incident (**klassevariabele**) |

**Tip:** Voer cellen **in volgorde** uit. Sectie 4 vergelijkt handmatige enumeratie met `LazyPropagation`.

### Stap 0 — Imports en instellingen

| Element | Betekenis |
|---------|-----------|
| `TARGET` | `Incident` — uitkomst / class label |
| `SEED` | Reproduceerbare steekproeven |
| `gum` / `gnb` | BN bouwen, inferentie, visualisatie |
| `bnvsbn` | Ground truth vs. geleerd netwerk |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import tempfile
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc

import pyagrum as gum
import pyagrum.skbn as skbn
import pyagrum.lib.notebook as gnb
import pyagrum.lib.bn_vs_bn as bnvsbn

TARGET = 'Incident'
SEED = 42
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

## Stap 1 — Variabelen en DAG (Taak 1)

Alle knopen zijn **binair** (`no`, `yes`) zodat je marginals en scenario's met pen en papier kunt controleren.

In [ ]:
states = {
    'Dissatisfaction': ['no', 'yes'],
    'BadBehaviour': ['no', 'yes'],
    'WeakTechControls': ['no', 'yes'],
    TARGET: ['no', 'yes'],
}

variable_descriptions = {
    'Dissatisfaction': 'Medewerker ontevreden (wortel).',
    'BadBehaviour': 'Zorgwekkend gedrag gezien; kind van Dissatisfaction.',
    'WeakTechControls': 'Technische controles zwak (wortel).',
    TARGET: 'Insider-incident — binair uitkomst / class label.',
}

pd.DataFrame([
  {'Variable': n, 'States': ', '.join(states[n]), 'Description': variable_descriptions[n]}
  for n in states
])

## Stap 2 — CPT's en noisy-OR (Taak 1 + RQ4)

### Handmatige parameters (uit de walkthrough)

| CPT | Waarde |
|-----|--------|
| $P(D{=}\text{yes})$ | 0,30 |
| $P(B{=}\text{yes} \mid D{=}\text{no})$ | 0,05 |
| $P(B{=}\text{yes} \mid D{=}\text{yes})$ | 0,50 |
| $P(T{=}\text{yes})$ | 0,25 |
| noisy-OR **leak** | 0,90 → $P(I{=}\text{no})$ zonder actieve oorzaak |
| inhibition **B** | 0,50 |
| inhibition **T** | 0,60 |

Incident-CPT (4 rijen):

| B | T | $P(I{=}\text{yes})$ |
|---|---|----------------------|
| no | no | 0,10 |
| yes | no | 0,55 |
| no | yes | 0,46 |
| yes | yes | 0,73 |

In [ ]:
def add_var(bn, name, labels):
    v = gum.LabelizedVariable(name, name, 0)
    for lab in labels:
        v.addLabel(lab)
    bn.add(v)


def noisy_or_p_yes(active_b, active_t, leak=0.90, inh_b=0.50, inh_t=0.60):
    """P(Incident=yes) voor binair noisy-OR; zelfde conventie als insider-notebook."""
    p_no = leak
    if active_b:
        p_no *= inh_b
    if active_t:
        p_no *= inh_t
    return 1.0 - p_no


def fill_incident_noisy_or(bn, leak=0.90, inh_b=0.50, inh_t=0.60):
    for b in states['BadBehaviour']:
        for t in states['WeakTechControls']:
            active_b = b == 'yes'
            active_t = t == 'yes'
            py = noisy_or_p_yes(active_b, active_t, leak=leak, inh_b=inh_b, inh_t=inh_t)
            bn.cpt(TARGET)[{'BadBehaviour': b, 'WeakTechControls': t}] = [1 - py, py]


def build_toy_bn(leak=0.90, inh_b=0.50, inh_t=0.60):
    bn = gum.BayesNet('Toy4Node')
    for name, labels in states.items():
        add_var(bn, name, labels)

    arcs = [
        ('Dissatisfaction', 'BadBehaviour'),
        ('BadBehaviour', TARGET),
        ('WeakTechControls', TARGET),
    ]
    for a, b in arcs:
        bn.addArc(a, b)

    bn.cpt('Dissatisfaction').fillWith([0.70, 0.30])
    bn.cpt('WeakTechControls').fillWith([0.75, 0.25])

    bn.cpt('BadBehaviour')[{'Dissatisfaction': 'no'}] = [0.95, 0.05]
    bn.cpt('BadBehaviour')[{'Dissatisfaction': 'yes'}] = [0.50, 0.50]

    fill_incident_noisy_or(bn, leak=leak, inh_b=inh_b, inh_t=inh_t)
    return bn


bn = build_toy_bn()
gnb.sideBySide(bn)

In [ ]:
# Incident-CPT expliciet tonen (handmatige controle)
inc_rows = []
for b in states['BadBehaviour']:
    for t in states['WeakTechControls']:
        py = float(bn.cpt(TARGET)[{'BadBehaviour': b, 'WeakTechControls': t, TARGET: 'yes'}])
        inc_rows.append({'BadBehaviour': b, 'WeakTechControls': t, 'P(Incident=yes)': round(py, 2)})
pd.DataFrame(inc_rows)

## Stap 3 — Inferentie en scenario's (Taak 1)

`LazyPropagation` berekent exacte posterior-kansen. We meten $P(\text{Incident}=\text{yes})$ onder herkenbare situaties — dezelfde logica als de scenario-tabel in het report.

In [ ]:
def p_incident_yes(model, evidence=None):
    ie = gum.LazyPropagation(model)
    if evidence:
        ie.setEvidence(evidence)
    ie.makeInference()
    return float(ie.posterior(TARGET)[{TARGET: 'yes'}])


ie = gum.LazyPropagation(bn)
ie.makeInference()
baseline = p_incident_yes(bn)
print(f'Baseline P({TARGET}=yes): {baseline:.3f}')

scenarios = {
    'Baseline': {},
    'Laag risico': {'BadBehaviour': 'no', 'WeakTechControls': 'no'},
    'Hoog risico': {'BadBehaviour': 'yes', 'WeakTechControls': 'yes'},
    'Alleen D=yes': {'Dissatisfaction': 'yes'},
    'Alleen B=yes': {'BadBehaviour': 'yes'},
}

rows = []
for name, ev in scenarios.items():
    rows.append({'Scenario': name, f'P({TARGET}=yes)': p_incident_yes(bn, ev)})

risk_df = pd.DataFrame(rows).round(3)
risk_df

In [ ]:
ax = sns.barplot(data=risk_df, x='Scenario', y=f'P({TARGET}=yes)', palette='viridis')
ax.set_ylim(0, 1)
ax.set_title('Incidentkans per scenario (toy 4-knopen BN)')
for i, v in enumerate(risk_df[f'P({TARGET}=yes)']):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

### Stap 3b — Marginals en conditionals

- **Marginals:** posterior zonder bewijs; controleer o.a. $P(B{=}\text{yes}) = 0{,}185$.
- **Conditionals:** $P(I{=}\text{yes} \mid B)$ moet stijgen t.o.v. baseline (0,265).

In [ ]:
# Marginals (baseline)
ie0 = gum.LazyPropagation(bn)
ie0.makeInference()
marginals = []
for name in states:
    post = ie0.posterior(name)
    for lab in states[name]:
        marginals.append({'Variable': name, 'State': lab, 'P': float(post[{name: lab}])})
marginal_df = pd.DataFrame(marginals).round(3)
marginal_df

In [ ]:
queries = [
    ('P(incident | B=no)', {'BadBehaviour': 'no'}),
    ('P(incident | B=yes)', {'BadBehaviour': 'yes'}),
    ('P(incident | T=yes)', {'WeakTechControls': 'yes'}),
]
cond_rows = []
for label, ev in queries:
    cond_rows.append({'Query': label, 'P(Incident=yes)': p_incident_yes(bn, ev)})
pd.DataFrame(cond_rows).round(3)

### Stap 3c — Sensitiviteit noisy-OR `leak`

Hogere `leak` → lagere basiskans op incident. Controle: blijft **laag < baseline < hoog**?

In [ ]:
leak_values = [0.85, 0.90, 0.95]
sens_rows = []
for leak in leak_values:
    bn_s = build_toy_bn(leak=leak)
    sens_rows.append({
        'leak': leak,
        'baseline': p_incident_yes(bn_s),
        'laag_risico': p_incident_yes(bn_s, {'BadBehaviour': 'no', 'WeakTechControls': 'no'}),
        'hoog_risico': p_incident_yes(bn_s, {'BadBehaviour': 'yes', 'WeakTechControls': 'yes'}),
    })
pd.DataFrame(sens_rows).round(3)

## Stap 4 — Handmatige enumeratie vs. pyAgrum

Hier controleren we of `LazyPropagation` overeenkomt met volledige sommatie over alle $2^3$ parent-configuraties. Handig om de rekenstappen uit de uitleg te verifiëren.

In [ ]:
# Handmatige enumeratie (zelfde parameters als build_toy_bn)
P_D = 0.30
P_B = {'no': 0.05, 'yes': 0.50}  # P(B=yes | D=no/yes)
P_T = 0.25
LEAK, INH_B, INH_T = 0.90, 0.50, 0.60


def manual_p_incident(evidence=None):
    evidence = evidence or {}
    num = den = 0.0
    for d_lab, p_d in [('no', 1 - P_D), ('yes', P_D)]:
        if 'Dissatisfaction' in evidence and d_lab != evidence['Dissatisfaction']:
            continue
        for b_lab in ['no', 'yes']:
            if 'BadBehaviour' in evidence and b_lab != evidence['BadBehaviour']:
                continue
            p_b = P_B[d_lab] if b_lab == 'yes' else 1 - P_B[d_lab]
            for t_lab in ['no', 'yes']:
                if 'WeakTechControls' in evidence and t_lab != evidence['WeakTechControls']:
                    continue
                p_t = P_T if t_lab == 'yes' else 1 - P_T
                w = p_d * p_b * p_t
                py = noisy_or_p_yes(b_lab == 'yes', t_lab == 'yes', LEAK, INH_B, INH_T)
                den += w
                num += w * py
    return num / den


def manual_p_B_yes():
    return (1 - P_D) * P_B['no'] + P_D * P_B['yes']


check_rows = []
for name, ev in scenarios.items():
    check_rows.append({
        'Scenario': name,
        'Handmatig': round(manual_p_incident(ev), 3),
        'pyAgrum': round(p_incident_yes(bn, ev or None), 3),
    })
check_df = pd.DataFrame(check_rows)
check_df['Verschil'] = (check_df['Handmatig'] - check_df['pyAgrum']).abs().round(6)
print('P(B=yes) handmatig:', round(manual_p_B_yes(), 3))
check_df

## Stap 5 — Taak 2.2a: synthetische data en structuur leren

1. `BNDatabaseGenerator` trekt cases uit het **handmatige** BN.
2. Hill Climbing (BIC) en MIIC leren structuur op subsets $n \in \{50, 100, 200\}$.
3. Vergelijk met ground-truth via skeleton/directed F1 en Hamming distance.

*In het grote model gebruiken jullie $n \in \{100,500,1000\}$ en $N=2000$; hier kleiner voor snelheid.*

In [ ]:
tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
tmp.close()
gen = gum.BNDatabaseGenerator(bn)
gen.drawSamples(1000)
gen.toCSV(tmp.name)
data = pd.read_csv(tmp.name)
os.unlink(tmp.name)

print('Synthetic data:', data.shape)
print('Incident rate:', data[TARGET].eq('yes').mean().round(3))
data.head(8)

In [ ]:
def learn_and_score(train_df, gt_bn):
    tf = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
    tf.close()
    train_df.to_csv(tf.name, index=False)

    learner_hc = gum.BNLearner(tf.name)
    learner_hc.useGreedyHillClimbing()
    learner_hc.useScoreBIC()
    learned_hc = learner_hc.learnBN()

    learner_miic = gum.BNLearner(tf.name)
    learner_miic.useMIIC()
    learned_miic = learner_miic.learnBN()
    os.unlink(tf.name)

    rows = []
    for name, model in [('HillClimb', learned_hc), ('MIIC', learned_miic)]:
        cmp = bnvsbn.GraphicalBNComparator(gt_bn, model)
        rows.append({
            'Algorithm': name,
            'Skeleton_F1': round(cmp.skeletonScores()['fscore'], 3),
            'Directed_F1': round(cmp.scores()['fscore'], 3),
            'Hamming': cmp.hamming()['hamming'],
        })
    return rows, learned_hc, learned_miic


struct_rows = []
for n in [50, 100, 200]:
    sample = data.sample(n=n, random_state=SEED)
    scores, _, _ = learn_and_score(sample, bn)
    for row in scores:
        struct_rows.append({'n': n, **row})

struct_df = pd.DataFrame(struct_rows)
struct_df

In [ ]:
# Visualisatie: ground truth vs. geleerd (n=200)
sample200 = data.sample(n=200, random_state=SEED)
_, learned_hc_200, learned_miic_200 = learn_and_score(sample200, bn)
print('Ground truth vs Hill Climbing (n=200)')
gnb.sideBySide(bn, learned_hc_200)

## Stap 6 — RQ4: noisy-OR monotoniciteit

Meer actieve oorzaken → hogere gemiddelde incidentkans. Twee actieve activators (B en T) moeten de hoogste rij geven.

In [ ]:
or_scenarios = [
    ('geen actief', {'BadBehaviour': 'no', 'WeakTechControls': 'no'}),
    ('alleen B', {'BadBehaviour': 'yes', 'WeakTechControls': 'no'}),
    ('alleen T', {'BadBehaviour': 'no', 'WeakTechControls': 'yes'}),
    ('B + T', {'BadBehaviour': 'yes', 'WeakTechControls': 'yes'}),
]
mono_rows = [{'Config': lbl, 'P(Incident=yes)': p_incident_yes(bn, ev)} for lbl, ev in or_scenarios]
mono_df = pd.DataFrame(mono_rows).round(3)
mono_df

In [ ]:
# Response curve: gemiddelde P(incident) bij k actieve oorzaken
curve_rows = []
for k in range(3):
    combos = [
        ev for ev in [
            {'BadBehaviour': b, 'WeakTechControls': t}
            for b in states['BadBehaviour']
            for t in states['WeakTechControls']
        ]
        if sum(v == 'yes' for v in ev.values()) == k
    ]
    vals = [p_incident_yes(bn, ev) for ev in combos]
    curve_rows.append({'active_causes': k, 'mean_P_incident': np.mean(vals)})
curve_df = pd.DataFrame(curve_rows).round(3)
curve_df

In [ ]:
plt.plot(curve_df['active_causes'], curve_df['mean_P_incident'], marker='o')
plt.xlabel('Aantal actieve noisy-OR oorzaken')
plt.ylabel('Gemiddelde P(Incident=yes)')
plt.title('Noisy-OR monotoniciteit (toy model)')
plt.xticks([0, 1, 2])
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Stap 7 — Taak 2.2b: classificatie

Drie modellen op dezelfde variabelen (zoals in de assignment):

| Model | Structuur | Parameters |
|-------|-----------|------------|
| **Original BN** | Expert-DAG (vast) | `learnParameters` op train |
| **Learned BN** | Hill Climbing op train | structuur + CPT's |
| **Naive Bayes** | ster naar `Incident` | geen keten D→B |

Train/test hier **40/40** (stratified op `Incident`) — in het echte report **100/100**.

In [ ]:
cls_pool = data.sample(n=200, random_state=SEED)
train_cls, test_cls = train_test_split(
    cls_pool,
    train_size=40,
    test_size=40,
    random_state=SEED,
    stratify=cls_pool[TARGET],
)
print('Classification split:', train_cls.shape, test_cls.shape)

train_csv = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
train_csv.close()
train_cls.to_csv(train_csv.name, index=False)


def infer_probs(model, test_df):
    probs = []
    for _, row in test_df.iterrows():
        ev = {c: row[c] for c in test_df.columns if c != TARGET}
        probs.append(p_incident_yes(model, ev))
    return np.array(probs)


def summarize(name, probs, y_true):
    y_bin = (y_true == 'yes').astype(int)
    fpr, tpr, _ = roc_curve(y_bin, probs)
    return {
        'Model': name,
        'AUC': round(auc(fpr, tpr), 3),
        'Accuracy@0.5': round(((probs >= 0.5).astype(int) == y_bin).mean(), 3),
    }


y_test = test_cls[TARGET].values

# Original BN: vaste structuur, parameters uit train
learner_true = gum.BNLearner(train_csv.name, bn)
learner_true.useSmoothingPrior()
bn_fit = learner_true.learnParameters(bn.dag())

# Learned BN
learner_hc = gum.BNLearner(train_csv.name)
learner_hc.useGreedyHillClimbing()
learner_hc.useScoreBIC()
bn_learned = learner_hc.learnBN()

# Naive Bayes
clf_naive = skbn.BNClassifier(learningMethod='NaiveBayes', scoringType='BIC')
clf_naive.fit(data=train_cls, targetName=TARGET)
nb_probs = clf_naive.predict_proba(test_cls.drop(columns=[TARGET]))[:, 1]
os.unlink(train_csv.name)

cls_rows = [
    summarize('Original BN (noisy-OR structuur)', infer_probs(bn_fit, test_cls), y_test),
    summarize('Learned BN (Hill Climbing)', infer_probs(bn_learned, test_cls), y_test),
    summarize('Naive Bayes', nb_probs, y_test),
]
pd.DataFrame(cls_rows)

In [ ]:
# ROC-curve (Taak 2.2b)
plt.figure(figsize=(7, 5))
for label, probs in [
    ('Original BN', infer_probs(bn_fit, test_cls)),
    ('Learned BN', infer_probs(bn_learned, test_cls)),
    ('Naive Bayes', nb_probs),
]:
    y_bin = (y_test == 'yes').astype(int)
    fpr, tpr, _ = roc_curve(y_bin, probs)
    plt.plot(fpr, tpr, label=f'{label} (AUC={auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title(f'ROC: classificatie {TARGET} (toy 4-knopen)')
plt.legend()
plt.tight_layout()
plt.show()

## Samenvatting — mapping naar het 8-knopen model

| Toy (4 knopen) | Insider-threat model |
|----------------|----------------------|
| `Dissatisfaction` | `JobDissatisfaction` |
| `BadBehaviour` | `ConcerningBehaviour` |
| `WeakTechControls` | `TechnicalControls` |
| `Incident` | `InsiderThreatIncident` |
| — | `SecurityAwareness` → `PolicyCompliance` → `PrivilegedAccess` |
| — | `MonitoringAlert` |

**Volgende stap:** open `insider_threat_bn_prototype.ipynb` voor de volledige pipeline met dezelfde secties op schaal.